In [ ]:

import os, json, time, re, sys, shutil
from datetime import datetime
from typing import Dict, Any, List
from openai import OpenAI
import uuid
import time
import shutil

# -----------------------------
# 0) Install/import deps
# -----------------------------
def _pip_install(pkgs: List[str]):
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import pandas as pd
except Exception:
    _pip_install(["pandas", "openpyxl"])
    import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    _pip_install(["python-dotenv"])
    from dotenv import load_dotenv

try:
    import tiktoken
except Exception:
    _pip_install(["tiktoken"])
    import tiktoken

try:
    from openai import OpenAI
except Exception:
    _pip_install(["openai"])
    from openai import OpenAI

# -----------------------------
# 1) Config
# -----------------------------
ROOT_DIR = os.getcwd()

# Source + working copy (resumable)
SOURCE_MESSAGES_PATH = os.path.join(ROOT_DIR, "messages.json")  # <-- read from this
RUN_BASENAME         = "12_51_Simple_NoContext"
WORK_MESSAGES_PATH   = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.json")
XLSX_OUT             = os.path.join(ROOT_DIR, f"{RUN_BASENAME}.xlsx")

# Stats file for resumable time + usage
STATS_PATH           = os.path.join(ROOT_DIR, f"{RUN_BASENAME}_run_stats.json")

ANNOTATION_MODEL     = "gpt-5.1"

# Output limit per request (tune as needed)
MAX_OUTPUT_TOKENS    = 2000

# For GPT-4.1: temperature is supported; set to 0 for determinism
#TEMPERATURE          = 0.0

# Pricing (per 1M tokens) — adjust if your contract differs
PRICE_INPUT_PER_1M   = 1.25
PRICE_CACHED_PER_1M  = 0.125
PRICE_OUTPUT_PER_1M  = 10.00

BINARY_FIELDS = [
    "DirectAddress", "Attachment", "SelfDisclose", "ReplySeeking",
    "CommRitual", "Monetary", "Backseat", "Emotes"
]
OTHER_FIELDS = ["Dimension"]

# -----------------------------
# 2) Prompt + schema placeholders (REPLACE YOURSELF)
# -----------------------------
DEVELOPER_PROMPT = r"""
You are labeling chat messages.

For each message, assign labels based only on the literal text. Do not infer intent beyond what is explicitly written.

Return ONLY a JSON array. No explanations.

Each array element corresponds to one input message and must contain ALL keys:
DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension.

For each label:
"Y" = present
""  = not present

Dimension:
"P" = positive tone
"N" = negative tone
Choose the more dominant tone if both appear.

Label definitions:

• DirectAddress:
Message talks directly to another person using "you", a name, or @mention.

• Attachment:
Message shows liking, love, or emotional warmth.

• SelfDisclose:
Speaker shares personal information about themselves.

• ReplySeeking:
Message asks a question or tries to get a response.

• CommRitual:
Message encourages shared or group behavior.

• Monetary:
Message mentions money, subscriptions, donations, or gifts.

• Backseat:
Message gives advice or instructions.

• Emotes:
Message includes emojis or emote-like text.

Example 1
Input:
"I love you <3"

Output:
{
  "DirectAddress": "Y",
  "Attachment": "Y",
  "SelfDisclose": "",
  "ReplySeeking": "",
  "CommRitual": "",
  "Monetary": "",
  "Backseat": "",
  "Emotes": "Y",
  "Dimension": "P"
}

Example 2
Input:
"Go left and use ult"

Output:
{
  "DirectAddress": "",
  "Attachment": "",
  "SelfDisclose": "",
  "ReplySeeking": "",
  "CommRitual": "",
  "Monetary": "",
  "Backseat": "Y",
  "Emotes": "",
  "Dimension": "P"
}

Example 3
Input:
"I’ve been sick all week lol"

Output:
{
  "DirectAddress": "",
  "Attachment": "",
  "SelfDisclose": "Y",
  "ReplySeeking": "",
  "CommRitual": "",
  "Monetary": "",
  "Backseat": "",
  "Emotes": "",
  "Dimension": "N"
}

Now label the following input messages.
Return ONLY a JSON array.
"""

# -----------------------------
# 7) Output schema (kept as in your code; not used in SDK call)
# -----------------------------
annotation_schema = {
    "name": "psi_annotation_batch",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "id": {"type": "integer"},
                        "DirectAddress": {"type": "string"},
                        "Attachment": {"type": "string"},
                        "SelfDisclose": {"type": "string"},
                        "ReplySeeking": {"type": "string"},
                        "CommRitual": {"type": "string"},
                        "Monetary": {"type": "string"},
                        "Backseat": {"type": "string"},
                        "Emotes": {"type": "string"},
                        "Dimension": {"type": "string"}
                    },
                    "required": [
                        "id","DirectAddress","Attachment","SelfDisclose",
                        "ReplySeeking","CommRitual","Monetary",
                        "Backseat","Emotes","Dimension"
                    ]
                }
            }
        },
        "required": ["items"]
    }
}

# -----------------------------
# 3) Env + client
# -----------------------------
load_dotenv(os.path.join(ROOT_DIR, ".env"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Ensure .env exists with OPENAI_API_KEY=...")

client = OpenAI()

# -----------------------------
# 4) Helpers
# -----------------------------
def now_ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def safe_write_json(path: str, obj: Any, retries: int = 10, sleep: float = 0.05):
    directory = os.path.dirname(path)
    os.makedirs(directory, exist_ok=True)

    tmp = f"{path}.{uuid.uuid4().hex}.tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    for attempt in range(retries):
        try:
            shutil.move(tmp, path)  # Windows-safe replace
            return
        except PermissionError:
            time.sleep(sleep)

    raise PermissionError(f"Could not replace {path} after {retries} retries")

def load_json_if_exists(path: str, default):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default

def ensure_working_copy():
    if not os.path.exists(SOURCE_MESSAGES_PATH):
        raise FileNotFoundError(f"messages.json not found at: {SOURCE_MESSAGES_PATH}")

    if not os.path.exists(WORK_MESSAGES_PATH):
        shutil.copy2(SOURCE_MESSAGES_PATH, WORK_MESSAGES_PATH)
        print(f"[{now_ts()}] Created working copy: {WORK_MESSAGES_PATH}")
    else:
        print(f"[{now_ts()}] Using existing working copy (resume): {WORK_MESSAGES_PATH}")

def load_messages() -> List[Dict[str, Any]]:
    with open(WORK_MESSAGES_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Working messages file must be a JSON array of message objects.")
    return data

def is_annotated(msg: Dict[str, Any]) -> bool:
    return str(msg.get("Annotated", "")).strip().lower() == "true"

def mark_annotated(msg: Dict[str, Any]):
    msg["Annotated"] = "true"

def ensure_fields(msg: Dict[str, Any]):
    for k in BINARY_FIELDS + OTHER_FIELDS:
        if k not in msg:
            msg[k] = ""
    for k in ["raw_msg", "msgTag", "streamer", "Annotated"]:
        if k not in msg:
            msg[k] = ""
    if msg["Annotated"] == "":
        msg["Annotated"] = "False"

def get_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-4.1")
    except Exception:
        return tiktoken.get_encoding("o200k_base")

ENC = get_encoding()

def tok_len(s: str) -> int:
    return len(ENC.encode(s or ""))

def usage_breakdown(resp) -> Dict[str, Any]:
    u = getattr(resp, "usage", None)
    if not u:
        return {}
    try:
        return u if isinstance(u, dict) else u.model_dump()
    except Exception:
        try:
            return dict(u)
        except Exception:
            return {"raw_usage": str(u)}

def enforce_binary(v: Any) -> str:
    return "Y" if str(v).strip().upper() == "Y" else ""

def enforce_dimension(v: Any, fallback: str = "P") -> str:
    s = str(v).strip().upper()
    if s in ("P", "POS", "POSITIVE"):
        return "P"
    if s in ("N", "NEG", "NEGATIVE"):
        return "N"
    return fallback

def clip_notes(text: str, max_words: int = 20) -> str:
    t = re.sub(r"\s+", " ", (text or "").strip())
    if not t:
        return ""
    words = t.split(" ")
    if len(words) <= max_words:
        return t
    return " ".join(words[:max_words]).rstrip() + "…"

def call_with_retries(kwargs, tries=6, base_sleep=1.5):
    last_err = None
    for attempt in range(1, tries + 1):
        try:
            return client.responses.create(**kwargs)
        except Exception as e:
            last_err = e
            sleep = base_sleep * (2 ** (attempt - 1))
            print(f"[{now_ts()}]   ERROR (attempt {attempt}/{tries}): {e}. Sleeping {sleep:.1f}s then retry...")
            time.sleep(sleep)
    raise last_err

def try_repair_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    t = t.strip()

    m_obj = re.search(r"\{.*\}", t, flags=re.DOTALL)
    m_arr = re.search(r"\[.*\]", t, flags=re.DOTALL)
    if m_arr and (not m_obj or len(m_arr.group(0)) >= len(m_obj.group(0))):
        t = m_arr.group(0)
    elif m_obj:
        t = m_obj.group(0)

    t = t.replace("\t", " ")
    t = re.sub(r",\s*([}\]])", r"\1", t)
    return t.strip()

def get_output_text(resp) -> str:
    if hasattr(resp, "output_text") and resp.output_text:
        return resp.output_text
    out = getattr(resp, "output", None) or []
    for item in out:
        if isinstance(item, dict) and item.get("type") == "message":
            for c in (item.get("content") or []):
                if isinstance(c, dict) and c.get("type") in ("output_text", "text"):
                    return c.get("text", "") or ""
    return ""

def reformat_to_json_only(bad_text: str) -> str:
    fix_prompt = (
        "You will be given text that is intended to be JSON but is invalid.\n"
        "Rewrite it into VALID JSON ONLY.\n"
        "Do not change meanings or labels. Do not omit any keys.\n"
        "Output ONLY valid JSON.\n\n"
        "TEXT:\n" + bad_text
    )
    r = client.responses.create(
        model=ANNOTATION_MODEL,
        input=[{"role": "user", "content": fix_prompt}],
        max_output_tokens=400,
        temperature=0.0,
    )
    return (get_output_text(r) or "").strip()

def parse_single_json(resp) -> Dict[str, Any]:
    text = get_output_text(resp)
    if not text:
        raise RuntimeError("Empty model output; cannot parse JSON.")

    t = try_repair_json(text)

    def _loads_or_none(s: str):
        try:
            return json.loads(s)
        except Exception:
            return None

    obj = _loads_or_none(t)

    # If parse failed, call the model once to reformat into valid JSON
    if obj is None:
        fixed = reformat_to_json_only(text)
        fixed2 = try_repair_json(fixed)
        obj = _loads_or_none(fixed2)
        if obj is None:
            raise RuntimeError(
                "JSON parse failed even after reformat.\n"
                f"---RAW_START---\n{text[:2000]}\n---RAW_END---\n"
                f"---FIXED_START---\n{fixed[:2000]}\n---FIXED_END---"
            )

    # Unwrap common shapes:
    # 1) {"items":[{...}]}  -> take first item
    if isinstance(obj, dict) and "items" in obj and isinstance(obj["items"], list):
        obj = obj["items"]

    # 2) [{...}] -> take first element
    if isinstance(obj, list):
        if len(obj) == 0:
            raise RuntimeError(f"Parsed JSON array is empty. text={text[:500]}")
        if not isinstance(obj[0], dict):
            raise RuntimeError(f"Expected list of objects, got list[{type(obj[0])}]. text={text[:500]}")
        obj = obj[0]

    if not isinstance(obj, dict):
        raise RuntimeError(f"Expected JSON object, got {type(obj)}. text={text[:500]}")

    # Normalize missing keys so downstream .get() isn't required
    for k in (BINARY_FIELDS + OTHER_FIELDS + ["Notes"]):
        obj.setdefault(k, "")

    return obj

def estimate_cost_usd(total_input: int, total_cached: int, total_output: int) -> float:
    cached = max(0, int(total_cached or 0))
    inp = max(0, int(total_input or 0))
    non_cached = max(0, inp - cached)

    cost = 0.0
    cost += (non_cached / 1_000_000.0) * PRICE_INPUT_PER_1M
    cost += (cached     / 1_000_000.0) * PRICE_CACHED_PER_1M
    cost += (max(0, int(total_output or 0)) / 1_000_000.0) * PRICE_OUTPUT_PER_1M
    return cost

def load_stats() -> Dict[str, Any]:
    stats = load_json_if_exists(STATS_PATH, default={})
    if not isinstance(stats, dict):
        stats = {}
    stats.setdefault("accumulated_seconds", 0.0)
    stats.setdefault("runs", 0)
    stats.setdefault("total_input_tokens", 0)
    stats.setdefault("total_output_tokens", 0)
    stats.setdefault("total_cached_tokens", 0)
    stats.setdefault("last_run_started_at", "")
    stats.setdefault("last_run_ended_at", "")
    return stats

def save_stats(stats: Dict[str, Any]):
    safe_write_json(STATS_PATH, stats)

def short_preview(s: str, n: int = 90) -> str:
    s = re.sub(r"\s+", " ", (s or "").strip())
    if len(s) <= n:
        return s
    return s[:n-1] + "…"

# -----------------------------
# 5) Setup working copy + load data + stats
# -----------------------------
ensure_working_copy()

messages = load_messages()
for m in messages:
    ensure_fields(m)

total = len(messages)
remaining = sum(1 for m in messages if not is_annotated(m))
print(f"[{now_ts()}] Loaded {total} messages from working copy. Remaining unannotated: {remaining}")

stats = load_stats()
stats["runs"] += 1
stats["last_run_started_at"] = now_ts()
save_stats(stats)

run_start = time.time()

# -----------------------------
# 6) Main loop (NO grouping; per-message calls; write+print every message)
# -----------------------------
newly_annotated = 0

for idx, msg in enumerate(messages):
    if is_annotated(msg):
        continue

    raw_msg = msg.get("raw_msg", "") or ""
    msgTag  = msg.get("msgTag", "") or ""

    # If empty message, still mark + persist to avoid stalling
    if not raw_msg.strip() and not msgTag.strip():
        mark_annotated(msg)
        newly_annotated += 1

        safe_write_json(WORK_MESSAGES_PATH, messages)
        elapsed = time.time() - run_start
        stats["accumulated_seconds"] += elapsed
        run_start = time.time()  # reset segment timer to keep writes safe if crash mid-loop
        stats["last_run_ended_at"] = now_ts()
        save_stats(stats)

        print(f"[{now_ts()}] #{idx} EMPTY -> marked Annotated=true | newly_annotated={newly_annotated}")
        continue

    user_payload = {"msgTag": msgTag, "raw_msg": raw_msg}

    kwargs = dict(
        model=ANNOTATION_MODEL,
        input=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": (
                "Annotate this SINGLE message.\n"
                "Return ONE JSON OBJECT (not an array, not {\"items\":...}) with keys:\n"
                "DirectAddress, Attachment, SelfDisclose, ReplySeeking, CommRitual, Monetary, Backseat, Emotes, Dimension, Notes.\n"
                "Use only \"Y\" or \"\" for the binary fields.\n"
                "Return ONLY valid JSON.\n\nINPUT_MESSAGE:\n"
                + json.dumps(user_payload, ensure_ascii=False)
            )}
        ],
        max_output_tokens=MAX_OUTPUT_TOKENS,
        reasoning={"effort": "low"},
        #temperature=TEMPERATURE,
    )

    t0 = time.time()
    resp = call_with_retries(kwargs)
    latency = time.time() - t0

    ud = usage_breakdown(resp)
    in_tok  = int(ud.get("input_tokens") or 0)
    out_tok = int(ud.get("output_tokens") or 0)
    cached_tok = 0
    try:
        cached_tok = int((ud.get("input_tokens_details") or {}).get("cached_tokens") or 0)
    except Exception:
        cached_tok = 0

    stats["total_input_tokens"]  += in_tok
    stats["total_output_tokens"] += out_tok
    stats["total_cached_tokens"] += cached_tok

    out_obj = parse_single_json(resp)

    # Apply fields robustly
    for f in BINARY_FIELDS:
        msg[f] = enforce_binary(out_obj.get(f, ""))

    if str(msg.get("msgTag", "")).strip().upper() == "USERNOTICE":
        msg["Monetary"] = "Y"

    raw_lower = raw_msg.lower()
    fallback_dim = "N" if re.search(r"\b(idiot|stupid|shut up|trash|hate|kill|die)\b", raw_lower) else "P"
    msg["Dimension"] = enforce_dimension(out_obj.get("Dimension", ""), fallback=fallback_dim)
    msg["Notes"] = clip_notes(out_obj.get("Notes", ""), max_words=20)

    mark_annotated(msg)
    newly_annotated += 1

    # Write JSON to disk AFTER EVERY MESSAGE
    safe_write_json(WORK_MESSAGES_PATH, messages)

    # Update resumable time AFTER EVERY MESSAGE
    elapsed_segment = time.time() - run_start
    stats["accumulated_seconds"] += elapsed_segment
    run_start = time.time()  # reset segment timer so a crash doesn't lose a lot of timing
    stats["last_run_ended_at"] = now_ts()

    # Save stats AFTER EVERY MESSAGE
    save_stats(stats)

    # Print update AFTER EVERY MESSAGE
    total_done = sum(1 for m in messages if is_annotated(m))
    cost_now = estimate_cost_usd(stats["total_input_tokens"], stats["total_cached_tokens"], stats["total_output_tokens"])

    print(
        f"[{now_ts()}] #{idx} streamer='{(msg.get('streamer') or '').strip()}' tag='{msgTag.strip()}' "
        f"msg='{short_preview(raw_msg, 110)}'\n"
        f"  -> DirectAddress={msg['DirectAddress'] or '-'} Attachment={msg['Attachment'] or '-'} "
        f"SelfDisclose={msg['SelfDisclose'] or '-'} ReplySeeking={msg['ReplySeeking'] or '-'} "
        f"CommRitual={msg['CommRitual'] or '-'} Monetary={msg['Monetary'] or '-'} "
        f"Backseat={msg['Backseat'] or '-'} Emotes={msg['Emotes'] or '-'} "
        f"Dimension={msg['Dimension'] or '-'} Notes='{msg['Notes']}'\n"
        f"  usage: in={in_tok} out={out_tok} cached_in={cached_tok} | latency={latency:.2f}s\n"
        f"  totals: annotated={total_done}/{total} | newly_annotated_this_run={newly_annotated} | est_cost_total=${cost_now:.6f}\n"
    )

# -----------------------------
# 7) Export XLSX
# -----------------------------
df = pd.DataFrame(messages)
front = ["streamer", "msgTag", "raw_msg", "Annotated"] + BINARY_FIELDS + ["Dimension", "Notes"]
cols = front + [c for c in df.columns if c not in front]
df = df[cols]
df.to_excel(XLSX_OUT, index=False)

# -----------------------------
# 8) Final summary
# -----------------------------
remaining = sum(1 for m in messages if not is_annotated(m))

total_input  = stats["total_input_tokens"]
total_output = stats["total_output_tokens"]
total_cached = stats["total_cached_tokens"]
total_cost = estimate_cost_usd(total_input, total_cached, total_output)

print(f"\n[{now_ts()}] DONE.")
print(f"[{now_ts()}] Total messages={total} | Newly annotated this run={newly_annotated} | Remaining={remaining}")
print(f"[{now_ts()}] Working JSON: {WORK_MESSAGES_PATH}")
print(f"[{now_ts()}] XLSX: {XLSX_OUT}")
print(f"[{now_ts()}] Resumable time (accumulated): {stats['accumulated_seconds']:.2f} seconds")
print(f"[{now_ts()}] Tokens total: input={total_input}, cached_input={total_cached}, output={total_output}")
print(f"[{now_ts()}] Estimated total cost (GPT-5.1): ${total_cost:.6f}")
print(f"[{now_ts()}] Run stats file: {STATS_PATH}")
